# Recovery from an erroring call, given to a small seed

`06` ended on the error that capped every one of its runs. A simulate call is fed one number in the unit the
question states it in -- 5.8 MPa written as `feed_pressure_bar 5.8`, 31.5 g/kg as `feed_nacl_mass_frac 31.5` --
the simulator errs, and the policy calls again: the same call, or another argument nudged, up to 20 times in a
row. 06's reference episodes never err, so its seeds had nothing to copy; 06 read it as "the label the next seed
has to carry". 06 also went 300 -> 600 -> 1,200 reference episodes each time PPO did not activate a dose, when
300 was already alive by its own rule.

This notebook changes the seed and nothing else -- the questions, the reward, the tools, the harness and PPO's
configuration are 06's:

* **Recovery episodes.** For a question with an input that needs converting, that input is left unconverted (the
  convert turn converts the others, the simulate turn passes the question's number as it stands), and when every
  call of that turn errs the simulator's real responses are kept. Then the fix: one line naming the input, its
  number and the unit the question gives it in (`simulate_ro failed: feed_pressure_bar 6.4, 6.6 and 6.8 are in
  MPa, as the question gives them, not bar. Converting them first.`), the convert_units call, the simulate turn
  again, the check turn and the answer. The slipped turns are context, never labelled: the seed is shown the error
  and taught the fix, not the slip.
* **A small seed, fixed in advance.** 300 episodes, 06's first alive dose: 200 of 06's reference episodes and 100
  recovery episodes of other questions, one epoch. The control is 06's 300 seed, retrained here.
* **The forced probe.** Every dev question with a slip starts from its slipped turns, errors included, and the
  policy goes on from there: how often it recovers from an error, apart from how often it makes one.
* **This machine.** The runs are on an Apple M3 Max (MPS), not 06's 16 GB CUDA card. Qwen3.5's linear-attention
  layers fall back to transformers' PyTorch rule there, whose triangular solve is 0.229 s of the rule's 0.234 s per
  layer; 07 replaces it with a recursive-doubling inverse on MPS only (same numbers to bf16's own noise).

**Results** (every number below is read from a run directory or a cached measurement; nothing is retrained here):

* **The seed carries the fix.** On the forced probe (each of the 144 dev questions with a slip started from its slipped
  turns, greedy) the seed recovers in **0.812** of the episodes against the control's 0.285 (reward 0.686 against
  0.401; exact repeats per episode 0.67 against 4.97). Sampled at T=1 on 120 dev questions it is weaker than the
  control overall (0.430 against 0.509; not yet alive by 06's rule, 14 of 16 answer classes) but recovers from 0.306
  of its erring episodes against the control's 0.113.
* **PPO activated it.** From the seed with 06's last configuration (120 steps sized for this machine), the series'
  first run whose greedy evaluation rose above step 0 and stayed there: **0.586 -> 0.658 -> 0.648 -> 0.673** at steps
  0, 40, 80 and 120 (60 dev questions; the best is the last step), decision right 0.550 -> 0.683, recovered after an
  erroring call 0.400 -> 0.500. In the training samples the reward went 0.470 (the warm-up: the seed itself) -> 0.552
  over the last 25 steps and recovery 0.372 -> 0.539, while the share of episodes with an erroring call stayed
  (0.53 -> 0.59): PPO grew the recovery, not the avoidance. No early stop tripped.
* **Held out so far:** the seed, greedy on the 120 test questions, 0.543. The run's checkpoint on test,
  holdout_shift, the forced probe and dev at T=1, and the control's test and holdout_shift, wait for the machine
  (notebook 08's PPO run has it first) and are added when run.

In [1]:
import collections, importlib.util, json, math, statistics as st, sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "07_recover_from_errors.py").exists() else Path("experiments/notebooks/smoke_test35")
# This notebook's own module (06's, copied and changed as its header says); loaded by path because the file
# name starts with a digit. Scoring, the recovery measures and the checks are pure Python: nothing below needs
# WaterTAP or a GPU.
spec = importlib.util.spec_from_file_location("nr07", HERE / "07_recover_from_errors.py")
nr = importlib.util.module_from_spec(spec)
sys.modules["nr07"] = nr
spec.loader.exec_module(nr)
RUNS, RUNS06 = nr.RUNS, HERE / "06_direction_and_compare_runs"
SEED, CTRL, PPO = "seed-n200-r100", "seed-n300-r0", "ppo-n200-r100-turn-s0"
splits = {s: nr.load_split(s) for s in nr.SPLITS}
cases = {c["id"]: c for cs in splits.values() for c in cs}


def rows_of(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def jload(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


def slip_of(case, call):
    """Which argument of an erroring simulate call is off, against the question's closest point: the question's
    own number in its own unit ("raw"), something else ("other"), or none (the call erred for another reason)."""
    if call["name"] != case["tool"]:
        return f"another tool ({call['name']})"
    args = call["arguments"] or {}
    best = None
    for p in case["points"]:
        off = [k for k, v in p["args"].items() if not nr._same_arg(args.get(k, nr.DEFAULTS[case["tool"]].get(k)), v)]
        if best is None or len(off) < len(best[0]):
            best = (off, p)
    off, p = best
    if not off:
        return "arguments right"
    tags = []
    for k in off:
        inp = case["inputs"].get(k) or p.get("inputs", {}).get(k)
        x = nr.parse_number(args.get(k))
        if inp and x is not None and abs(x - inp["user_value"]) <= 0.005 * abs(inp["user_value"]):
            tags.append(f"raw {k} [{inp['unit']}]")
        else:
            tags.append(f"other {k}")
    return "; ".join(tags)


def after_error(rows):
    """What episodes did after their first erroring simulate call."""
    out, units = collections.Counter(), collections.Counter()
    for ep in rows:
        case, calls = cases[ep["id"]], ep["calls"]
        out["episodes"] += 1
        first = next((i for i, c in enumerate(calls) if c["name"] not in nr.LOCAL_TOOLS and c["error"]), None)
        if first is None:
            continue
        c0, rs = calls[first], nr.recovery_stats(calls)
        out["erred"] += 1
        out["recovered"] += rs["recovered"]
        out["3+ erroring simulate calls in a row"] += rs["streak"] >= 3
        if "score" in ep:
            out["decision right"] += bool(ep["score"]["decision_correct"])
        later = [c for c in calls if c["turn"] > c0["turn"]]
        if not later:
            nxt = "the answer (or nothing)"
        elif later[0]["name"] == nr.CONVERT:
            nxt = "a convert_units call"
        elif later[0]["name"] == nr.COMPARE:
            nxt = "check_limits on what it had"
        elif later[0]["name"] == c0["name"] and later[0]["arguments"] == c0["arguments"]:
            nxt = "the same call again"
        elif later[0]["name"] == c0["name"]:
            nxt = "the simulate call, an argument changed"
        else:
            nxt = "another call"
        out["next: " + nxt] += 1
        units[slip_of(case, c0)] += 1
    return out, units


print({s: len(c) for s, c in splits.items()}, "| questions read from", nr.QDATA.name, "| tools:", ", ".join(nr.ALL_TOOLS))

{'train': 1200, 'dev': 240, 'test': 240, 'holdout_shift': 120} | questions read from 06_direction_and_compare_data | tools: simulate_ro, simulate_swro_system, convert_units, check_limits


## What 06's seeds did after an erroring call

Every dev probe of 06's seeds, sampled at T=1, read for one thing: the episodes whose simulate call erred, what the
policy wrote next, and whether a clean simulate call ever followed. The argument that was off is read against the
question's closest point: almost always one input carries the question's own number in the question's own unit --
the conversion skipped -- not a wrong conversion.

In [2]:
PROBES06 = [("06: 300 episodes", RUNS06 / "label-seed-n300/epoch1/dev-T1-stop.episodes.jsonl"),
            ("06: 600 episodes (its seed)", RUNS06 / "label-seed-n600/epoch1/dev-T1-x2-stop.episodes.jsonl"),
            ("06: 1200 episodes", RUNS06 / "label-seed-n1200/epoch1/dev-T1-x2-stop.episodes.jsonl")]
NEXT = ["next: the same call again", "next: the simulate call, an argument changed", "next: a convert_units call",
        "next: check_limits on what it had", "next: the answer (or nothing)", "next: another call"]
print(f"{'dev, T=1':30} {'episodes':>8} {'erred':>6} {'recovered':>9} {'decision right':>14} {'3+ in a row':>11}")
slips06 = collections.Counter()
for label, path in PROBES06:
    out, units = after_error(rows_of(path))
    slips06 += units
    e = out["erred"]
    print(f"{label:30} {out['episodes']:>8} {e:>6} {out['recovered']:>4} ({out['recovered'] / e:.2f}) "
          f"{out['decision right']:>7} ({out['decision right'] / e:.2f}) {out['3+ erroring simulate calls in a row']:>6}")
    print("   what came next:", ", ".join(f"{k[6:]} {out[k]}" for k in NEXT if out[k]))
raw = sum(v for k, v in slips06.items() if k.startswith("raw") and ";" not in k)
print(f"\nthe argument that was off, first erroring call, three seeds pooled ({sum(slips06.values())} calls): "
      f"one input in the question's own unit {raw}")
for k, v in slips06.most_common(14):
    print(f"   {v:4d}  {k}")

dev, T=1                       episodes  erred recovered decision right 3+ in a row
06: 300 episodes                    240    108   17 (0.16)      15 (0.14)     92
   what came next: the same call again 31, the simulate call, an argument changed 18, a convert_units call 21, check_limits on what it had 16, the answer (or nothing) 1, another call 21
06: 600 episodes (its seed)         480    135   24 (0.18)      20 (0.15)    111
   what came next: the same call again 37, the simulate call, an argument changed 21, a convert_units call 31, check_limits on what it had 24, the answer (or nothing) 4, another call 18
06: 1200 episodes                   480     41    5 (0.12)       7 (0.17)     34
   what came next: the same call again 17, the simulate call, an argument changed 5, a convert_units call 4, check_limits on what it had 9, the answer (or nothing) 2, another call 4

the argument that was off, first erroring call, three seeds pooled (284 calls): one input in the question's own unit 1

## The recovery episodes

`reference --recover 1` builds one per question that has an input converted alone: the input is drawn per
question, and an input whose slip the simulator runs on some call -- a silently wrong number, not an error to
recover from -- is passed over for the next. The simulator does not always say which input is wrong: a salinity in
g/kg trips the server's own validation ("feed_nacl_mass_frac must be between 0 and 1"), A in LMH/bar or a pressure in
kPa trips a variable bound that names the variable, but a pressure in MPa, a flow in kg/h or an area in ft² gives only
"Unit model fs.unit failed to initialize", and the plant model a paragraph about infeasible inputs. The fix line is
the same for all of them because it reads the question and the call, not the error: the number still in the
question's unit. What a recovery episode costs in reward is the erroring calls themselves (they enter the calls
credit and the efficiency count, as a policy's would).

In [3]:
checks = jload(HERE / "07_recover_from_errors_checks.json") or {}
harness = jload(HERE / "07_recover_from_errors_harness_checks.json") or {}
for split, r in checks.get("recovery", {}).items():
    print(f"{split}: {r['recovery_episodes']} recovery episodes of {r['questions']} questions "
          f"({r['with an input converted alone']} have an input converted alone); reward mean {r['reward_mean']} "
          f"min {r['reward_min']} (the same questions' reference episodes: {r['reference_reward_mean']})")
    print("   by family:", r["by_family"])
    print("   the simulator's answer:", r["error_kind"])
    print("   inputs tried before one erred on every call:", r["inputs_tried_before_one_erred"])
    if split == "train":
        print("   slipped input:", dict(list(r["slipped_input"].items())[:12]), "...")
if harness.get("recovery"):
    h = harness["recovery"]
    print(f"\nharness checks on {h['n']} dev recovery episodes: calls that survive the XML round trip {h['calls_parse_roundtrip']}/{h['n']}; "
          f"token ids equal to the chat template's rendering (the fix line in the assistant message's content) {h['token_ids_equal_template']}/{h['n']}")
for name in ("train_reference_policy_tokens", "train_recovery_policy_tokens"):
    if name in harness:
        t = harness[name]
        print(f"policy tokens of the {t['episodes']} {name.split('_')[1]} trajectories (slipped turns included): median {t['median']}, "
              f"p90 {t['p90']}, max {t['max']}; budget {t['budget']:,}, over it {t['over_budget']}")
for err in checks.get("recovery", {}).get("train", {}).get("example_errors", {}).items():
    print(f"\n[{err[0]}] {err[1]}")
path = nr.recovery_path("train")
if path.exists():
    ep = next(r for r in rows_of(path) if r["slip"]["error_kind"] == "failed to initialize" and len(r["turns"]) == 5)
    case = cases[ep["id"]]
    print(f"\n== {ep['id']} (slipped: {ep['slip']['param']} in {ep['slip']['unit']})\n{case['prompt'][:420]}...\n")
    for t, turn in enumerate(ep["turns"]):
        mark = "labelled" if turn.get("label", True) else "context (not labelled)"
        print(f"-- turn {t + 1}, {mark}: {len(turn['calls'])} {turn['calls'][0]['name']} call(s)")
        if turn.get("text"):
            print("   " + turn["text"])
        print("   " + nr.call_block(turn["calls"][0]["name"], turn["calls"][0]["arguments"]).replace("\n", " ")[:230])
        print("   -> " + turn["results"][0].replace("\n", " ")[:150])
    print("-- the answer turn, labelled:\n" + ep["answer_reasoned"])

train: 697 recovery episodes of 1200 questions (698 have an input converted alone); reward mean 0.9425 min 0.8911 (the same questions' reference episodes: 0.9859)
   by family: {'ro_scan_pressure': 124, 'ro_check': 123, 'swro_check': 121, 'ro_membrane': 116, 'swro_option': 113, 'ro_scan_area': 100}
   the simulator's answer: {'validation (names the input)': 282, 'out of bounds (names the variable)': 177, 'failed to initialize': 146, 'other': 92}
   inputs tried before one erred on every call: {'2': 56, '1': 632, '3': 9}
   slipped input: {'feed_pressure_bar [kPa]': 72, 'feed_pressure_bar [MPa]': 67, 'feed_nacl_mass_frac [g/kg]': 58, 'feed_nacl_mass_frac [mg/kg]': 58, 'feed_nacl_mass_frac [permille]': 54, 'feed_nacl_mass_frac [pct]': 51, 'p1_pressure_bar [kPa]': 42, 'p1_efficiency [pct]': 42, 'feed_flow_m3_s [L/s]': 39, 'feed_tds_g_L [mg/L]': 30, 'membrane_area_m2 [ft2]': 30, 'p1_pressure_bar [MPa]': 30} ...
dev: 144 recovery episodes of 240 questions (144 have an input converted alone)

## This machine: PPO on Apple silicon

06 ran on a 16 GB CUDA card (under WSL). 07 runs on an Apple M3 Max (40-core GPU, 64 GB of memory shared with the
CPU) through PyTorch's MPS backend, with the simulators in arm64 worker processes (IDAES' darwin-aarch64 solvers;
the same results as the data to 1e-9, `05`'s `mlx-check`). Nothing else changes, except one function.

Qwen3.5-0.8B has 18 linear-attention (Gated DeltaNet) layers. Without the fla / causal-conv1d kernels -- CUDA-only --
transformers runs them through a PyTorch reference rule, and on MPS that rule's `torch.linalg.solve_triangular` is
0.229 s of its 0.234 s per layer on a 6k-token episode, and the slicing inside its chunk scan makes the backward copy
whole tensors once per chunk. `mps_chunk_gated_delta_rule` is the same computation with the triangular system
inverted by recursive doubling (LAPACK trtri's algorithm, batched matmuls) and the scan on views taken once. Against
float64 on the real UT systems of an episode its inverse errs by 3.3e-7, solve_triangular by 3.1e-7; on the model's
log-probs it sits where transformers' own rule sits against the model in float32. It is installed on MPS only.

In [4]:
dev_ = jload(HERE / "07_recover_from_errors_device.json")
if dev_:
    print(f"{dev_['device']} on {dev_['system']} {dev_['machine']}, torch {dev_['torch']}; one train reference episode, "
          f"{dev_['episode_tokens']:,} tokens ({dev_['labelled_tokens']} labelled)")
    for k, label in (("forward_s", "forward, log-probs at the labelled tokens"),
                     ("forward_backward_checkpointed_s", "forward + backward, checkpointed (the actor's pass)")):
        v = dev_[k]
        print(f"   {label:52} transformers' rule {v['transformers']:6.2f}s" + (f"   07's rule {v['07']:6.2f}s" if "07" in v else ""))
    for name, e in dev_["logprob_vs_float32"].items():
        print(f"   log-probs, bf16 with {name:>12}'s rule, against the model in float32: max {e['max']:.4f}, mean {e['mean']:.5f}")
    g = dev_["generate_8_prompts"]
    print(f"   generation, 8 prompts of {g['prompt_tokens']:,} tokens ({g['rule']}'s rule): 1 new token {g['1_new_s']}s, 256 new {g['256_new_s']}s")
# 06's card, for scale: the seconds per PPO step of its slow turn run (generation, and the whole step)
m06 = rows_of(RUNS06 / "ppo-n600-open-turn-lr2e6-s0/metrics.jsonl")
act06 = [r for r in m06 if not r["critic_warmup"]]
print(f"\n06's CUDA card, ppo-n600-open-turn-lr2e6-s0: a step after the warm-up {st.median(r['step_seconds'] for r in act06):.0f}s median "
      f"(generation {st.median(r['gen_seconds'] for r in act06):.0f}s); the warm-up steps (no actor update) {st.median(r['step_seconds'] for r in m06 if r['critic_warmup']):.0f}s")
m07 = RUNS / PPO / "metrics.jsonl"
if m07.exists():
    r7 = rows_of(m07)
    a7 = [r for r in r7 if not r["critic_warmup"]]
    w7 = [r for r in r7 if r["critic_warmup"]]
    print(f"07 on this machine, {PPO}: " + (f"a step after the warm-up {st.median(r['step_seconds'] for r in a7):.0f}s median "
          f"(generation {st.median(r['gen_seconds'] for r in a7):.0f}s); " if a7 else "") +
          (f"the warm-up steps {st.median(r['step_seconds'] for r in w7):.0f}s" if w7 else ""))

mps on Darwin arm64, torch 2.14.1; one train reference episode, 6,803 tokens (1106 labelled)
   forward, log-probs at the labelled tokens            transformers' rule   5.82s   07's rule   1.35s
   forward + backward, checkpointed (the actor's pass)  transformers' rule  25.63s   07's rule   6.03s
   log-probs, bf16 with transformers's rule, against the model in float32: max 0.2038, mean 0.00319
   log-probs, bf16 with           07's rule, against the model in float32: max 0.1794, mean 0.00321
   generation, 8 prompts of 2,794 tokens (07's rule): 1 new token 4.31s, 256 new 10.98s

06's CUDA card, ppo-n600-open-turn-lr2e6-s0: a step after the warm-up 182s median (generation 114s); the warm-up steps (no actor update) 121s
07 on this machine, ppo-n200-r100-turn-s0: a step after the warm-up 474s median (generation 148s); the warm-up steps 170s


## The seed and the control

Both are 300 episodes, one epoch of SFT at lr 1e-4, LoRA r16, every token of the calls and the answer labelled. The
seed is the first 200 of 06's reference episodes (06's shuffle, so they are the first 200 of 06's own 300) and 100
recovery episodes of other train questions; the control is 06's 300, retrained on this machine. On Apple silicon a
batch of 16 sampled episodes takes minutes, so the probes are sized down from 06's: 120 dev questions -- the same
120 for every policy, 06's seeds read on the same 120 -- sampled once at T=1 (judged by `alive`, with one more label:
an erroring simulate call, then a clean one), and the forced probe, greedy, on all 144 dev questions with a slip, each
started from its slipped turns. The probes fold the adapter into the weights for generation.

In [5]:
import random
DEV120 = {c["id"] for c in random.Random(0).sample(splits["dev"], 120)}     # probe --limit 120's questions
TEST120 = {c["id"] for c in random.Random(0).sample(splits["test"], 120)}


def recov_line(o):
    r = o["recovery"]
    f = lambda x: "  -  " if x is None else f"{x:.3f}"  # noqa: E731
    return (f"n {o['n']:3d} reward {o['reward']:.3f} decision {o['decision_correct']:.3f} evidence {o['evidence']:.3f} "
            f"answers {o['answer_found']:.3f} | erred {r['sim_erred']:.3f} recovered {f(r['recovered_given_error'])} "
            f"decision|erred {f(r['decision_given_error'])} decision|clean {f(r['decision_given_no_error'])} "
            f"repeats/ep {r['repeats_per_episode']:.2f} 3+ in a row {r['streak3_share']:.3f}")


def summary_of(path, ids=None):
    """A probe's measures from its episodes, on the questions in ``ids`` when given (06's probes ran on all of a
    split; 07's on a fixed 120 of dev and test)."""
    ep_path = Path(str(path).replace(".json", ".episodes.jsonl"))
    if not ep_path.exists():
        return None
    rows = [r for r in rows_of(ep_path) if ids is None or r["id"] in ids]
    if not rows:
        return None
    sc = [r["score"] for r in rows]
    n = len(rows)
    return {"n": n, "reward": sum(x["total"] for x in sc) / n, "decision_correct": sum(x["decision_correct"] for x in sc) / n,
            "evidence": sum(x["evidence"] for x in sc) / n,
            "answer_found": sum(nr.extract_answer(r["final_text"])[0] is not None for r in rows) / n,
            "recovery": nr.recovery_block([nr.recovery_stats(r["calls"]) for r in rows], sc)}


print("dev, the 120 questions, each sampled once at T=1:")
for label, path in [("06: the 300 seed (06's card)", RUNS06 / "label-seed-n300/epoch1/dev-T1-stop.json"),
                    ("06: the 600 seed (06's card)", RUNS06 / "label-seed-n600/epoch1/dev-T1-stop.json"),
                    (f"07: control, {CTRL}", RUNS / CTRL / "epoch1/dev-T1-n120-stop.json"),
                    (f"07: seed, {SEED}", RUNS / SEED / "epoch1/dev-T1-n120-stop.json")]:
    o = summary_of(path, DEV120)
    print(f"  {label:34} " + (recov_line(o) if o else "(not run yet)"))
print("\nthe forced probe, greedy: the dev questions with a slip, each started from its slipped turns")
for run in (CTRL, SEED):
    o = summary_of(RUNS / run / "epoch1/dev-T0-stop-forced.json")
    print(f"  {run:16} " + (recov_line(o) if o else "(not run yet)"))
for run in (CTRL, SEED):
    p = RUNS / run / "epoch1/dev-T1-n120-stop.episodes.jsonl"
    if p.exists():
        L = nr.labels_of(rows_of(p), {c["id"]: c for c in splits["dev"]})
        print(f"\nalive, {run}: {'ALIVE' if L['alive'] else 'NOT YET ALIVE'}; answer classes right at least once "
              f"{sum(1 for v in L['classes'].values() if v[0] > 0)}/{len(L['classes'])}")
        for k in nr.LABELS:
            print(f"   {k:46} {L['hit'][k]:>4} {L['rate'][k]:.3f}" + ("   (of the episodes that erred)" if k == nr.LABELS[-1] else ""))

dev, the 120 questions, each sampled once at T=1:
  06: the 300 seed (06's card)       n 120 reward 0.518 decision 0.450 evidence 0.492 answers 0.700 | erred 0.442 recovered 0.132 decision|erred 0.132 decision|clean 0.701 repeats/ep 1.38 3+ in a row 0.392
  06: the 600 seed (06's card)       n 120 reward 0.706 decision 0.675 evidence 0.667 answers 0.775 | erred 0.267 recovered 0.250 decision|erred 0.188 decision|clean 0.852 repeats/ep 0.68 3+ in a row 0.217
  07: control, seed-n300-r0          n 120 reward 0.509 decision 0.475 evidence 0.458 answers 0.692 | erred 0.442 recovered 0.113 decision|erred 0.189 decision|clean 0.701 repeats/ep 1.54 3+ in a row 0.317
  07: seed, seed-n200-r100           n 120 reward 0.430 decision 0.342 evidence 0.367 answers 0.575 | erred 0.600 recovered 0.306 decision|erred 0.139 decision|clean 0.646 repeats/ep 0.59 3+ in a row 0.392

the forced probe, greedy: the dev questions with a slip, each started from its slipped turns
  seed-n300-r0     n 144 reward 

## PPO from the seed

06's last configuration: the turn as the unit of credit (one assistant message is one action, lambda 0.95 per turn),
the actor's step 2e-6 (the step at which 06's turn run did not run away), KL 0.01 to the seed, critic warm-up 25
steps, the value head on layer 23 warm-started at the seed's sampled dev reward, the tool gate off and the
answer-collapse rule on. Sized for this machine, where a step takes about 7 minutes against 3 on 06's card: greedy
evaluations on 60 dev questions every 40 steps (06: 120 every 25), and 120 steps for a first look (06: 200). What the
run is read for: whether the evaluations rise above step 0 (06's criterion, which no 06 run met), and whether the
recovery the seed planted grows -- the share of erring episodes that run a clean simulate call afterwards, in the
training samples of every step and in the evaluations.

In [6]:
def rolling(xs, k=10):
    out = []
    for i in range(len(xs)):
        w = [x for x in xs[max(0, i - k + 1): i + 1] if x is not None]
        out.append(sum(w) / len(w) if w else None)
    return out


def report_run(run, png):
    d = RUNS / run
    if not (d / "metrics.jsonl").exists():
        print(f"{run}: not run yet")
        return None
    m, ev = rows_of(d / "metrics.jsonl"), rows_of(d / "eval.jsonl") if (d / "eval.jsonl").exists() else []
    ending = jload(d / "ending.json")
    print(f"{run}: {len(m)} steps" + (f"; ended at step {ending['stopped_at']} ({ending['reason']}), best evaluation step "
                                     f"{ending['best_step']} ({ending['best_reward']:.4f})" if ending else " (running)"))
    warm = [r for r in m if r["critic_warmup"]]
    act = [r for r in m if not r["critic_warmup"]]
    for label, rows in (("warm-up (the seed, sampled)", warm), ("actor steps", act), ("last 25 steps", act[-25:])):
        if not rows:
            continue
        mean = lambda k: sum(r[k] for r in rows if r.get(k) is not None) / max(1, sum(r.get(k) is not None for r in rows))  # noqa: E731
        print(f"   {label:28} reward {mean('reward_mean'):.3f} decision {mean('decision_correct'):.3f} evidence {mean('evidence'):.3f} "
              f"answered {mean('answered'):.3f} | erred {mean('rec_sim_erred'):.3f} recovered {mean('rec_recovered_given_error'):.3f} "
              f"repeats/ep {mean('rec_repeats_per_episode'):.2f} | kl {mean('kl_mean'):.4f}")
    if ev:
        print(f"   greedy evaluations, {(jload(d / 'config.json') or {}).get('eval_cases', '?')} dev questions:")
        for r in ev:
            rc = r.get("recovery", {})
            g = rc.get("recovered_given_error")
            print(f"     step {r['step']:>4}: reward {r['reward']:.4f} decision {r['decision_correct']:.3f} evidence {r['evidence']:.3f} "
                  f"answers {r['answer_found']:.3f} calls errored {r['calls_errored']:.3f} | erred {rc.get('sim_erred', float('nan')):.3f} "
                  f"recovered {'-' if g is None else f'{g:.3f}'}")
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return m
    steps = [r["step"] for r in m]
    panels = [("reward_mean", "sampled reward"), ("decision_correct", "decision right"), ("evidence", "proving simulations"),
              ("rec_sim_erred", "episodes with an erroring simulate call"),
              ("rec_recovered_given_error", "of those, a clean simulate call after"), ("rec_repeats_per_episode", "exact repeats per episode"),
              ("kl_mean", "KL to the seed, per token"), ("value_ev", "critic: explained variance")]
    fig, axes = plt.subplots(2, 4, figsize=(18, 7))
    for ax, (k, title) in zip(axes.flat, panels):
        ys = [r.get(k) for r in m]
        ax.plot(steps, [y if y is not None else float("nan") for y in ys], ".", alpha=0.25, color="tab:blue")
        ax.plot(steps, [y if y is not None else float("nan") for y in rolling(ys)], "-", color="tab:blue")
        if k == "value_ev":
            ax.plot(steps, rolling([r.get("value_ev_position") for r in m]), "-", color="tab:gray", label="position clock")
            ax.legend()
        ax.axvline(25, color="k", lw=0.5, ls=":")
        ax.set_title(title, fontsize=10)
    for r in ev:
        axes.flat[0].plot(r["step"], r["reward"], "s", color="tab:red")
        axes.flat[1].plot(r["step"], r["decision_correct"], "s", color="tab:red")
        g = r.get("recovery", {}).get("recovered_given_error")
        if g is not None:
            axes.flat[4].plot(r["step"], g, "s", color="tab:red")
    fig.suptitle(f"{run}: training samples (blue, rolling 10) and greedy dev evaluations (red); the warm-up ends at the dotted line")
    fig.tight_layout()
    fig.savefig(HERE / png, dpi=90)
    plt.close(fig)
    print(f"   figure: {png}")
    return m


M_PPO = report_run(PPO, "07_recover_from_errors_curves.png")

ppo-n200-r100-turn-s0: 120 steps; ended at step 120 (step budget 120 reached), best evaluation step 120 (0.6730)
   warm-up (the seed, sampled)  reward 0.470 decision 0.400 evidence 0.395 answered 0.710 | erred 0.530 recovered 0.372 repeats/ep 0.31 | kl 0.0000
   actor steps                  reward 0.538 decision 0.516 evidence 0.467 answered 0.749 | erred 0.529 recovered 0.452 repeats/ep 0.44 | kl 0.0019
   last 25 steps                reward 0.552 decision 0.540 evidence 0.480 answered 0.755 | erred 0.585 recovered 0.539 repeats/ep 0.29 | kl 0.0030
   greedy evaluations, 60 dev questions:
     step    0: reward 0.5856 decision 0.550 evidence 0.500 answers 0.633 calls errored 0.568 | erred 0.583 recovered 0.400
     step   40: reward 0.6580 decision 0.633 evidence 0.600 answers 0.717 calls errored 0.496 | erred 0.483 recovered 0.448
     step   80: reward 0.6479 decision 0.633 evidence 0.583 answers 0.683 calls errored 0.514 | erred 0.533 recovered 0.438
     step  120: reward 0.6730 

   figure: 07_recover_from_errors_curves.png


## Held out

Greedy on 120 test questions (the same 120 for every policy; 06's seeds read on the same 120) and all 120 of
holdout_shift for the seed, the control and the run's checkpoints (its best, when that is not step 0, and its last
adapter); the run's checkpoints also through the forced probe and sampled on the 120 dev questions at T=1, as the
seed was.

The run ended with its best evaluation at its last step, so its best checkpoint and its last adapter are the same
weights (step 120), probed once and kept in `probes-step120/`. These probes were paused after the seed's test probe so
that notebook 08's PPO run could have the GPU (on Apple silicon two GPU jobs slow each other ~2x); rows reading
"(not run yet)" are those; `07_recover_from_errors_eval_heldout.sh` resumes where it stopped.

In [7]:
rows_ = [(f"06: the 600 seed (06's card)", RUNS06 / "label-seed-n600/epoch1", "06"),
         (f"06: the 1200 seed (06's card)", RUNS06 / "label-seed-n1200/epoch1", "06"),
         (f"07: control {CTRL}", RUNS / CTRL / "epoch1", "07"), (f"07: seed {SEED}", RUNS / SEED / "epoch1", "07")]
# The run ended with its best evaluation at its last step (120): adapter-best, adapter and adapter-step120 are the same
# weights, probed once; the probes are kept in probes-step120/ (adapter directories are not kept in git).
if (RUNS / PPO / "probes-step120").exists():
    rows_.append((f"07: {PPO} step 120 (best = last)", RUNS / PPO / "probes-step120", "07"))
for split, ids, tag07 in (("test", TEST120, "test-T0-n120-stop"), ("holdout_shift", None, "holdout_shift-T0-stop")):
    print(f"{split}, greedy" + (" (the 120 questions)" if ids else "") + ":")
    for label, d, nb_ in rows_:
        o = summary_of(d / (f"{split}-T0-stop.json" if nb_ == "06" else f"{tag07}.json"), ids)
        print(f"  {label:44} " + (recov_line(o) if o else "(not run yet)"))
print("\ndev, the forced probe (greedy) and the 120 questions at T=1, the run's checkpoint against the seed:")
for label, d, nb_ in rows_[3:]:
    for tag in ("dev-T0-stop-forced", "dev-T1-n120-stop"):
        o = summary_of(d / f"{tag}.json")
        print(f"  {label:44} {tag:20} " + (recov_line(o) if o else "(not run yet)"))

test, greedy (the 120 questions):
  06: the 600 seed (06's card)                 n 120 reward 0.809 decision 0.800 evidence 0.758 answers 0.833 | erred 0.183 recovered 0.136 decision|erred 0.091 decision|clean 0.959 repeats/ep 1.24 3+ in a row 0.175
  06: the 1200 seed (06's card)                n 120 reward 0.926 decision 0.908 evidence 0.958 answers 0.975 | erred 0.025 recovered 0.667 decision|erred 0.000 decision|clean 0.932 repeats/ep 0.19 3+ in a row 0.025
  07: control seed-n300-r0                     (not run yet)
  07: seed seed-n200-r100                      n 120 reward 0.543 decision 0.508 evidence 0.442 answers 0.700 | erred 0.517 recovered 0.452 decision|erred 0.274 decision|clean 0.759 repeats/ep 0.24 3+ in a row 0.333
  07: ppo-n200-r100-turn-s0 step 120 (best = last) (not run yet)
holdout_shift, greedy:
  06: the 600 seed (06's card)                 n 120 reward 0.802 decision 0.767 evidence 0.775 answers 0.892 | erred 0.158 recovered 0.211 decision|erred 0.211 decision

## Summary

Set before the runs, the three readings of the PPO run:

1. **06's criterion -- met.** Every greedy evaluation on the dev questions is above step 0: 0.586 -> 0.658 (step 40),
   0.648 (80), **0.673** (120, the best), with the decision right 0.550 -> 0.683 and the proving simulations 0.500 ->
   0.633. No 06 run had one. (60 questions per evaluation; the evaluations do not keep their episodes, so there is no
   paired interval for them.)
2. **The recovery label -- grew.** Of the training episodes with an erroring simulate call, the share that ran a clean
   one afterwards: 0.372 in the warm-up (the seed itself, sampled) against **0.539** over the run's last 25 steps; in
   the greedy evaluations 0.400 -> 0.500. The share of episodes that err did not fall (0.530 -> 0.585): PPO made the
   policy fix its errors more often, not make fewer. The forced probe at the run's checkpoint (the seed: 0.812) is
   waiting for the machine.
3. **The sampled policy -- waiting.** The 120 dev questions at T=1, the run's checkpoint against the seed (0.430),
   paired per question: the probe is queued behind notebook 08's PPO run.

Also left: PPO from the control (deferred: notebook 08 replaces the critic, from this run's configuration, and has the
machine first).
